In [3]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from gensim.models import KeyedVectors
import kagglehub

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
df = pd.read_csv("/kaggle/input/datasets/kritanjalijain/amazon-reviews/train.csv",header = None, names=['rating','title','review'])
df = df.sample(n = 100000,random_state = 42)
reviews = df['review'].dropna().astype(str)
df

,rating,title,review
2079998,1,Expensive Junk,This product consists of a piece of thin flexi...
1443106,1,Toast too dark,"Even on the lowest setting, the toast is too d..."
3463669,2,Excellent imagery...dumbed down story,I enjoyed this disc. The video is stunning. I ...
2914699,1,Are we pretending everyone is married?,The authors pretend that parents neither die n...
1603231,1,Not worth your time,"Might as well just use a knife, this product h..."
...,...,...,...
144445,1,A Good Intro book but not a GREAT one!,"Well, I must admit, this book has great photos..."
1230897,1,Not pleased.,The Firm Lower body sculpt DVD did not meet my...
656995,2,Son loves these!,I buy these for my 8 yr. old son. I had to loo...
1834995,1,Terribad,"Firstly I have a 2009 Dodge Dakota, so maybe i..."


In [5]:
stop_words = set(stopwords.words('english'))
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[a-z\s]','',text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return tokens
tokenized_reviews = reviews.apply(preprocess)    

In [6]:
import gensim.downloader as  api
model = api.load("glove-wiki-gigaword-300")
print("vector size:",model.vector_size)

vector size: 300


In [7]:
def sentence_vector(tokens):
    valid_words = [w for w in tokens if w in model.key_to_index]
    
    if len(valid_words) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(model[valid_words], axis=0)

sentence_embeddings = np.vstack(
    tokenized_reviews.apply(sentence_vector).values
)

print("Sentence embedding shape:", sentence_embeddings.shape)

Sentence embedding shape: (100000, 300)


In [8]:
# Example 1
result = model.most_similar(
    positive=['excellent', 'good'],
    negative=['bad'],
    topn=5
)
print("good - bad + excellent =", result)

# Example 2
result = model.most_similar(
    positive=['fast', 'delivery'],
    negative=['late'],
    topn=5
)
print(result)


good - bad + excellent = [('superb', 0.5926157832145691), ('terrific', 0.5719386339187622), ('wonderful', 0.5324779748916626), ('quality', 0.5093278288841248), ('great', 0.5008903741836548)]
[('faster', 0.47034794092178345), ('deliveries', 0.46527913212776184), ('bowler', 0.4512387812137604), ('slow', 0.4483119249343872), ('pace', 0.44423744082450867)]


In [16]:
from tqdm import tqdm
skipped = 0
correct = 0
total = 0
with open("/kaggle/input/datasets/vidwathkumar/quextions/questions-words.txt", "r") as f:
    for line in tqdm(f):
        if line.startswith(":"):
            continue
        
        w1, w2, w3, w4 = line.strip().lower().split()
        
        if any(w not in model.key_to_index for w in [w1, w2, w3, w4]):
            skipped += 1
            continue
        
        prediction = model.most_similar(
            positive=[w2, w3],
            negative=[w1],
            topn=1
        )[0][0]
        
        if prediction == w4:
            correct += 1
        
        total += 1

print("Correct:", correct)
print("Total evaluated:", total)
print("Skipped (OOV):", skipped)
print("Accuracy:", correct / total)


19558it [08:33, 38.06it/s]

Correct: 14020
Total evaluated: 19544
Skipped (OOV): 0
Accuracy: 0.7173557101923864
